In [14]:
# to check the time stamps from video and epoch info
from pynwb import NWBHDF5IO
import pandas as pd
import numpy as np

path_to_file = "/mnt/spyglass-data/raw/na5520260516.nwb"

# Open NWB file read-only and close it before any h5py edits below.
with NWBHDF5IO(path_to_file, "r", load_namespaces=True) as io:
    nwb = io.read()

    # Get epoch intervals as a DataFrame
    epochs_df = nwb.intervals['epochs'].to_dataframe()
    print("Original epoch times:")
    print(epochs_df)

    # Get video start/stop times
    video_ts = nwb.processing['video_files']['video'].time_series
    video_df = []
    for key in video_ts:
        t = video_ts[key].get_timestamps()
        video_df.append({
            "video": key,
            "start_time": t[0],
            "end_time": t[-1]
        })
    video_df = pd.DataFrame(video_df)
    print("\nVideo times:")
    print(video_df)


Original epoch times:
      start_time     stop_time        tags
id                                        
0   1.778919e+09  1.778920e+09  [01_sleep]
1   1.778920e+09  1.778921e+09    [02_run]
2   1.778921e+09  1.778922e+09  [03_sleep]
3   1.778922e+09  1.778923e+09    [04_run]
4   1.778923e+09  1.778924e+09  [05_sleep]
5   1.778924e+09  1.778925e+09    [06_run]
6   1.778925e+09  1.778926e+09  [07_sleep]
7   1.778926e+09  1.778927e+09    [08_run]
8   1.778928e+09  1.778928e+09  [09_sleep]

Video times:
                        video    start_time      end_time
0  20260516_na55_02_run.1.mp4  1.778920e+09  1.778921e+09
1  20260516_na55_04_run.1.mp4  1.778922e+09  1.778923e+09
2  20260516_na55_06_run.1.mp4  1.778924e+09  1.778925e+09
3  20260516_na55_08_run.1.mp4  1.778926e+09  1.778927e+09


## Use this code only if the sleep videos are not included

In [6]:
# for mapping epoch timestamps to video timestamps
import h5py
import numpy as np
import pandas as pd
from pynwb import NWBHDF5IO
import re

# -------------------------------
# PARAMETERS
# -------------------------------
path = path_to_file
include_mapped_videos = False
comment = "\n[INFO] Epoch timestamps were aligned to video timestamps."

# -------------------------------
# HELPER FUNCTIONS
# -------------------------------
def confirm(prompt):
    """Ask user for yes/no input."""
    x = input(prompt + " (y/n): ").strip().lower()
    return x == "y"

def read_hdf5_text(dataset):
    value = dataset[()]
    if isinstance(value, bytes):
        return value.decode("utf-8")
    return str(value)

def replace_hdf5_scalar_text(group, name, value):
    """Replace a scalar UTF-8 text dataset instead of assigning in-place."""
    if name in group:
        del group[name]
    group.create_dataset(name, data=value, dtype=h5py.string_dtype(encoding="utf-8"))


def get_epoch_tag_number_and_type(tags):
    """
    tags example: ['02_run'] or ['01_sleep']
    returns: '02_run'
    """
    if isinstance(tags, (list, tuple, np.ndarray)):
        tag = str(tags[0])
    else:
        tag = str(tags)

    m = re.search(r"(\d{2}_(?:run|sleep))", tag)
    if m is None:
        return None

    return m.group(1)
def get_epoch_run_tag(tags):
    if isinstance(tags, (list, tuple, np.ndarray)):
        tag = str(tags[0])
    else:
        tag = str(tags)

    m = re.search(r"(\d{2}_run)", tag)
    if m is None:
        return None

    return m.group(1)


def get_video_run_tag(video_name):
    m = re.search(r"(\d{2}_run)", str(video_name))
    if m is None:
        return None

    return m.group(1)

# -------------------------------
# READ CURRENT EPOCH AND VIDEO TIMES
# -------------------------------
io = NWBHDF5IO(path, "r")
try:
    nwbfile = io.read()

    epochs_df = nwbfile.intervals["epochs"].to_dataframe().copy()

    video_module = nwbfile.processing["video_files"]["video"].time_series
    video_keys = sorted(video_module.keys())

    if not include_mapped_videos:
        video_keys = [key for key in video_keys if not key.endswith("_mapped")]

    video_rows = []
    for key in video_keys:
        ts = video_module[key]
        t_array = np.asarray(ts.get_timestamps())

        video_rows.append({
            "video": key,
            "video_start_time": float(t_array[0]),
            "video_stop_time": float(t_array[-1]),
            "frames": len(t_array),
        })

    video_df = pd.DataFrame(video_rows)
    epochs_df["epoch_tag"] = epochs_df["tags"].apply(get_epoch_run_tag)
    video_df["video_tag"] = video_df["video"].apply(get_video_run_tag)
    # -------------------------------
    # PRINT TABLES FOR MANUAL CHECK
    # -------------------------------
    print("=== Current Epoch Timestamp Table ===")
    print(epochs_df)

    print("\n=== Video Timestamp Table ===")
    print(video_df)

    n_epochs = len(epochs_df)
    n_videos = len(video_df)
    epoch_ids = epochs_df.index.to_numpy()

    map_rows = []

    for epoch_row, (epoch_id, epoch_row_data) in enumerate(epochs_df.iterrows()):
        epoch_tag = epoch_row_data["epoch_tag"]

        if epoch_tag is None:
            print(f"[WARNING] Could not read tag for epoch id {epoch_id}. Skipping.")
            continue

        matching_videos = video_df[video_df["video_tag"] == epoch_tag]

        if matching_videos.empty:
            print(f"[WARNING] No video found for epoch tag {epoch_tag}. This epoch will stay unchanged.")
            continue

        if len(matching_videos) > 1:
            print(f"[WARNING] Multiple videos found for epoch tag {epoch_tag}. Using first one.")

        video_row = matching_videos.iloc[0]

        map_rows.append({
            "epoch_row": epoch_row,
            "epoch_id": epoch_id,
            "epoch_tag": epoch_tag,
            "video": video_row["video"],
            "current_epoch_start": epoch_row_data["start_time"],
            "video_start_time": video_row["video_start_time"],
            "start_diff_sec": video_row["video_start_time"] - epoch_row_data["start_time"],
            "current_epoch_stop": epoch_row_data["stop_time"],
            "video_stop_time": video_row["video_stop_time"],
            "stop_diff_sec": video_row["video_stop_time"] - epoch_row_data["stop_time"],
            "frames": video_row["frames"],
        })

    comparison_df = pd.DataFrame(map_rows)

    print("\n=== Manual Check: Epochs Matched By Tag ===")
    print(comparison_df)

    if comparison_df.empty:
        raise ValueError("No matching epoch/video pairs found. Check epoch tags and video names.")


finally:
    io.close()

# -------------------------------
# ASK PERMISSION AFTER PRINTING TABLES
# -------------------------------
if confirm("\nDo you want to overwrite epoch timestamps with the video timestamps shown above?"):
    with h5py.File(path, "a") as f:
        print("\nCurrent epoch start times:", f["intervals"]["epochs"]["start_time"][:])
        print("Current epoch stop times:", f["intervals"]["epochs"]["stop_time"][:])

        old_start_times = f["intervals"]["epochs"]["start_time"][:]
        old_stop_times = f["intervals"]["epochs"]["stop_time"][:]

        for _, row in comparison_df.iterrows():
            epoch_row = int(row["epoch_row"])
            old_start_times[epoch_row] = row["video_start_time"]
            old_stop_times[epoch_row] = row["video_stop_time"]

        f["intervals"]["epochs"]["start_time"][:] = old_start_times
        f["intervals"]["epochs"]["stop_time"][:] = old_stop_times

        print("New epoch start times:", f["intervals"]["epochs"]["start_time"][:])
        print("New epoch stop times:", f["intervals"]["epochs"]["stop_time"][:])

        desc = read_hdf5_text(f["session_description"])
        if comment not in desc:
            replace_hdf5_scalar_text(f, "session_description", desc.rstrip() + comment)
            print("\nAdded mapping comment to session_description.")
        else:
            print("\nMapping comment already exists in session_description.")

    io = NWBHDF5IO(path, "r")
    try:
        nwbfile = io.read()

        print("\n=== Verified Epoch Table After Mapping ===")
        print(nwbfile.intervals["epochs"].to_dataframe())

        print("\n=== Verified session_description ===")
        print(nwbfile.session_description)

    finally:
        io.close()

    print("\nDone. File is closed and can be opened elsewhere.")

else:
    print("No mapping applied. File is closed and can be opened elsewhere.")


=== Current Epoch Timestamp Table ===
      start_time     stop_time        tags epoch_tag
id                                                  
0   1.756875e+09  1.756876e+09  [01_sleep]      None
1   1.756876e+09  1.756877e+09    [02_run]    02_run
2   1.756877e+09  1.756878e+09  [03_sleep]      None
3   1.756878e+09  1.756879e+09    [04_run]    04_run
4   1.756879e+09  1.756880e+09  [05_sleep]      None
5   1.756880e+09  1.756881e+09    [06_run]    06_run
6   1.756881e+09  1.756881e+09  [07_sleep]      None

=== Video Timestamp Table ===
                        video  video_start_time  video_stop_time  frames  \
0  20250903_na55_02_run.1.mp4      1.756441e+09     1.756442e+09  112720   
1  20250903_na55_04_run.1.mp4      1.756443e+09     1.756444e+09  114434   
2  20250903_na55_06_run.1.mp4      1.756445e+09     1.756446e+09  112786   

  video_tag  
0    02_run  
1    04_run  
2    06_run  
[WARNING] Could not read tag for epoch id 0. Skipping.
[WARNING] Could not read tag for epoch

## Use the code if both sleep and run videos are included

In [3]:
# for mapping epoch timestamps to video timestamps
import h5py
import numpy as np
import pandas as pd
from pynwb import NWBHDF5IO

# -------------------------------
# PARAMETERS
# -------------------------------
path = path_to_file
include_mapped_videos = False
comment = "\n[INFO] Epoch timestamps were aligned to video timestamps."

# -------------------------------
# HELPER FUNCTIONS
# -------------------------------
def confirm(prompt):
    """Ask user for yes/no input."""
    x = input(prompt + " (y/n): ").strip().lower()
    return x == "y"

def read_hdf5_text(dataset):
    value = dataset[()]
    if isinstance(value, bytes):
        return value.decode("utf-8")
    return str(value)

def replace_hdf5_scalar_text(group, name, value):
    """Replace a scalar UTF-8 text dataset instead of assigning in-place."""
    if name in group:
        del group[name]
    group.create_dataset(name, data=value, dtype=h5py.string_dtype(encoding="utf-8"))

# -------------------------------
# READ CURRENT EPOCH AND VIDEO TIMES
# -------------------------------
io = NWBHDF5IO(path, "r")
try:
    nwbfile = io.read()

    epochs_df = nwbfile.intervals["epochs"].to_dataframe().copy()

    video_module = nwbfile.processing["video_files"]["video"].time_series
    video_keys = sorted(video_module.keys())

    if not include_mapped_videos:
        video_keys = [key for key in video_keys if not key.endswith("_mapped")]

    video_rows = []
    for key in video_keys:
        ts = video_module[key]
        t_array = np.asarray(ts.get_timestamps())

        video_rows.append({
            "video": key,
            "video_start_time": float(t_array[0]),
            "video_stop_time": float(t_array[-1]),
            "frames": len(t_array),
        })

    video_df = pd.DataFrame(video_rows)

    # -------------------------------
    # PRINT TABLES FOR MANUAL CHECK
    # -------------------------------
    print("=== Current Epoch Timestamp Table ===")
    print(epochs_df)

    print("\n=== Video Timestamp Table ===")
    print(video_df)

    if len(epochs_df) != len(video_df):
        raise ValueError(
            f"Cannot safely map epochs to videos: found {len(epochs_df)} epochs "
            f"but {len(video_df)} videos. Check the file before mapping."
        )

    comparison_df = pd.DataFrame({
        "epoch_id": epochs_df.index.to_numpy(),
        "video": video_df["video"].to_numpy(),

        "current_epoch_start": epochs_df["start_time"].to_numpy(),
        "video_start_time": video_df["video_start_time"].to_numpy(),
        "start_diff_sec": (
            video_df["video_start_time"].to_numpy()
            - epochs_df["start_time"].to_numpy()
        ),

        "current_epoch_stop": epochs_df["stop_time"].to_numpy(),
        "video_stop_time": video_df["video_stop_time"].to_numpy(),
        "stop_diff_sec": (
            video_df["video_stop_time"].to_numpy()
            - epochs_df["stop_time"].to_numpy()
        ),

        "frames": video_df["frames"].to_numpy(),
    })

    print("\n=== Manual Check: Current Epochs vs Video Times ===")
    print(comparison_df)

    print("\nIf you continue, the epoch start/stop times will be replaced with:")
    print(video_df[["video", "video_start_time", "video_stop_time"]])

    new_start_times = video_df["video_start_time"].to_numpy(dtype=float)
    new_stop_times = video_df["video_stop_time"].to_numpy(dtype=float)

finally:
    io.close()

# -------------------------------
# ASK PERMISSION AFTER PRINTING TABLES
# -------------------------------
if confirm("\nDo you want to overwrite epoch timestamps with the video timestamps shown above?"):
    with h5py.File(path, "a") as f:
        print("\nCurrent epoch start times:", f["intervals"]["epochs"]["start_time"][:])
        print("Current epoch stop times:", f["intervals"]["epochs"]["stop_time"][:])

        f["intervals"]["epochs"]["start_time"][:] = new_start_times
        f["intervals"]["epochs"]["stop_time"][:] = new_stop_times

        print("New epoch start times:", f["intervals"]["epochs"]["start_time"][:])
        print("New epoch stop times:", f["intervals"]["epochs"]["stop_time"][:])

        desc = read_hdf5_text(f["session_description"])
        if comment not in desc:
            replace_hdf5_scalar_text(f, "session_description", desc.rstrip() + comment)
            print("\nAdded mapping comment to session_description.")
        else:
            print("\nMapping comment already exists in session_description.")

    io = NWBHDF5IO(path, "r")
    try:
        nwbfile = io.read()

        print("\n=== Verified Epoch Table After Mapping ===")
        print(nwbfile.intervals["epochs"].to_dataframe())

        print("\n=== Verified session_description ===")
        print(nwbfile.session_description)

    finally:
        io.close()

    print("\nDone. File is closed and can be opened elsewhere.")

else:
    print("No mapping applied. File is closed and can be opened elsewhere.")

=== Current Epoch Timestamp Table ===
      start_time     stop_time        tags
id                                        
0   1.756875e+09  1.756876e+09  [01_sleep]
1   1.756876e+09  1.756877e+09    [02_run]
2   1.756877e+09  1.756878e+09  [03_sleep]
3   1.756878e+09  1.756879e+09    [04_run]
4   1.756879e+09  1.756880e+09  [05_sleep]
5   1.756880e+09  1.756881e+09    [06_run]
6   1.756881e+09  1.756881e+09  [07_sleep]

=== Video Timestamp Table ===
                        video  video_start_time  video_stop_time  frames
0  20250903_na55_02_run.1.mp4      1.756441e+09     1.756442e+09  112720
1  20250903_na55_04_run.1.mp4      1.756443e+09     1.756444e+09  114434
2  20250903_na55_06_run.1.mp4      1.756445e+09     1.756446e+09  112786


ValueError: Cannot safely map epochs to videos: found 7 epochs but 3 videos. Check the file before mapping.